<a href="https://colab.research.google.com/github/sherjahong1r/Machine-Learning-Lessons/blob/main/16_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NEW SECTION**

# **4. Transformer modeli uchun ma’lumotlarni tayyorlash**

## **4.1. Ma’lumotlar to‘plamini yuklash**

### datasets kutubxonasi yordamida 1.2 qismda tayyorlangan DataFrame’dan Dataset obyekti yarating. Ma’lumotlarni train (2500 ta) va test (500 ta) qismlariga ajrating (seed=123).

## **4.2. Matnni tokenizatsiya qilish**





### distilbert-base-uncased tokenizer’ini yuklang.



### Matnlarni tokenizatsiya qiluvchi, padding va truncation qo‘llaydigan preprocess_function yarating (max_length=256).



###Funksiyani train va test to‘plamlariga map metodi orqali qo‘llang.

In [4]:
!wget https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv


--2026-03-21 08:14:12--  https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66212309 (63M) [text/plain]
Saving to: ‘IMDB-Dataset.csv.1’

IMDB-Dataset.csv.1  100%[===================>]  63.14M   292MB/s    in 0.2s    

2026-03-21 08:14:14 (292 MB/s) - ‘IMDB-Dataset.csv.1’ saved [66212309/66212309]



In [5]:
import pandas as pd

df = pd.read_csv("IMDB-Dataset.csv.1")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
positive_reviews = df[df['sentiment'] == 'positive'].sample(1500, random_state=101)
negative_reviews = df[df['sentiment'] == 'negative'].sample(1500, random_state=101)

df = pd.concat([positive_reviews, negative_reviews]).sample(frac=1, random_state=101).reset_index(drop=True)
df
# random_state=101: Tasodifiy tanlovni har doim bir xil natija berishini ta'minlaydi. Ya'ni,
# kodni necha marta ishga tushirilsa ham, bir xil tasodifiy namuna olinadi.
# frac=1: DataFrame'dagi barcha qatorlarni (100%) tanlaydi, lekin ularni tasodifiy tartibda aralashtirib beradi.

,review,sentiment
0,What an amazingly funny and original show. The...,positive
1,Those two main characters Erkan and Stefan are...,positive
2,The biggest and most disconcerting things of t...,negative
3,Dig! I would say to anyone even if you don't l...,positive
4,I have seen several Yul Brynner films--yet thi...,positive
...,...,...
2995,"While all of the Fleischer/Famous Studios ""Sup...",positive
2996,"This movie was chosen, quite frankly as a pig ...",negative
2997,The subsequent two seasons of this original se...,positive
2998,"This movie is not worth seeing, at least not a...",negative


In [7]:
df.shape

(3000, 2)

In [8]:
!pip install --upgrade huggingface_hub
from datasets import Dataset

dataset = Dataset.from_pandas(df)
small_train = dataset.shuffle(seed=123 ).select(range(2500))
small_test = dataset.shuffle(seed=123).select(range(500))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Define label mappings
label2id = {'negative': 0, 'positive': 1}
id2label = {0: 'negative', 1: 'positive'}

def preprocess_function(examples):
    tokenized_inputs = tokenizer(examples["review"], truncation=True, padding='max_length', max_length=256)
    # Convert string labels to numerical IDs
    tokenized_inputs["labels"] = [label2id[label] for label in examples["sentiment"]]
    return tokenized_inputs

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [10]:
print(small_train.column_names)
print(small_test.column_names)

['review', 'sentiment']
['review', 'sentiment']


# **5. Transformer modelini o‘qitish va baholash**

## **5.1. Modelni yuklash va sozlash**


### distilbert-base-uncased asosida AutoModelForSequenceClassification modelini yuklang


### TrainingArguments sozlamalarini yarating: num_train_epochs=3, logging_steps=100.


### Trainer obyektini yarating.

## **5.2. Modelni o‘qitish va baholash**


### trainer.train() metodi bilan modelni shug‘ullantiring.


### trainer.evaluate() metodini chaqirib, modelning eval_loss ko‘rsatkichini tahlil qiling.

In [11]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification
import os
import torch

# Ensure label2id and id2label are defined as in cell YjznBuEQZF38, or reuse them if they are global
# For robustness, we redefine them here for this specific cell's execution context if not already global.
label2id = {'negative': 0, 'positive': 1}
id2label = {0: 'negative', 1: 'positive'}

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label2id), # Use the number of defined labels
    id2label=id2label,
    label2id=label2id
)

# Modelni aniq CPU ga o'tkazish (Agar yuqoridagi muhit o'zgaruvchisi yetarli bo'lmasa)
# model.to('cpu') kodini faqat CUDA mavjud bo'lganda ishlatish maqsadga muvofiq.
# Agar CUDA_VISIBLE_DEVICES="" o'rnatilgan bo'lsa, PyTorch CUDA ni ko'rmaydi.
# Lekin agar model hali ham GPU da bo'lsa, uni CPU ga o'tkazish kerak.
if torch.cuda.is_available() and model.device.type == 'cuda':
    model.to('cpu')

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=100,
    report_to='none'
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

trainer.evaluate()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.438655
200,0.330213
300,0.183566
400,0.114033


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.03413411229848862,
 'eval_runtime': 3.6204,
 'eval_samples_per_second': 138.108,
 'eval_steps_per_second': 8.839,
 'epoch': 3.0}

# **Modelni baholash va sinovdan o‘tkazish**

In [13]:
texts = [
    "I hated this movie","Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi.",
    "I waited for this film for a long time, but I didn't like it at all.",
    " Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan.",
    "The cast is amazing, and the plot is very interesting.",
    "Filmning vizual effektlari yaxshi, ammo hikoyasi zaif.",
    "The film's visual effects are good, but the story is weak.",

    "this movie is a masterpiece", "a complete waste of time",
    "This movie was an absolute masterpiece, highly recommend watching it!"
]

In [14]:
import torch
import torch.nn.functional as F

In [15]:
# Inputs ni to'g'ridan-to'g'ri CPU ga yuborish
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=256).to('cpu')

# Modelni ham CPU ga o'tkazilganligiga ishonch hosil qilish
if torch.cuda.is_available() and model.device.type == 'cuda':
    model.to('cpu')

with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1)

label_map = {0: "negative", 1: 'possitive'}

for text, pred, prob in zip(texts, predictions, probs):
    print(text, label_map[pred.item()], prob[pred.item()].item())

I hated this movie negative 0.9829014539718628
Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi. possitive 0.7205780148506165
I waited for this film for a long time, but I didn't like it at all. negative 0.993954598903656
 Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan. possitive 0.8348077535629272
The cast is amazing, and the plot is very interesting. possitive 0.9924542307853699
Filmning vizual effektlari yaxshi, ammo hikoyasi zaif. possitive 0.8724076747894287
The film's visual effects are good, but the story is weak. negative 0.9934912323951721
this movie is a masterpiece possitive 0.9967181086540222
a complete waste of time negative 0.9958042502403259
This movie was an absolute masterpiece, highly recommend watching it! possitive 0.9973658919334412


In [16]:
import torch

def predict_sentiment(text):
  # Inputs ni CPU ga yuborish, chunki model ham CPU da ishlamoqda
  inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=256).to('cpu')

  with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1).item()

  return "Ijobiy" if predictions == 1 else "Salbiy"

  # Bu predict_sentiment funksiyasi berilgan matnning ('text') sentimentini (ijobiy yoki salbiy) aniqlash uchun
  # ishlatiladi. U matnni tokenizatsiya qiladi, modeldan o'tkazib tahmin ehtimolliklarini oladi, eng yuqori
  # ehtimolga ega sinfni tanlaydi va natijani 'Ijobiy' yoki 'Salbiy' ko'rinishida qaytaradi.

In [17]:

import gradio as gr

interface = gr.Interface(
    fn=predict_sentiment,
    inputs="text",
    outputs="text",
    title="Kino uchun izoh tahlili",
    description="Sizning fikiringiz o'rganilib kino uchun `ijobiy` yoki `salbiy` ekanligi aniqlanadi. `text` qismiga faqat inglizcha gap kiritish mumkin va uni natijasi uchun `Submit` qiling!"
)

interface.launch(share=True)

# Bu kod Gradio kutubxonasi yordamida kichik veb-interfeys yaratadi. predict_sentiment funksiyasini kirish sifatida
# 'text' olib, natijani 'text' ko'rinishida chiqaradi. title va description interfeysning sarlavhasi va tavsifini
# belgilaydi. interface.launch(share=True) esa bu veb-interfeyni ishga tushirib, uni umumiy foydalanish uchun havola yaratadi

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be06386335fad4ff51.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:

torch.save(model.state_dict(), 'Sentiment_model.pth')
# Bu kod model.state_dict() yordamida modelning o'qitilgan parametrlarini (vazn va boshqa konfiguratsiyalarini)
# oladi va ularni 'model.pth' nomli faylga saqlaydi. Bu modelni keyinchalik qayta yuklash va ishlatish imkonini beradi.

In [19]:
model.load_state_dict(torch.load('Sentiment_model.pth'))
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
